# **Book 1**

In [ ]:
!pip install -q langchain langchain-community llama-parse llama-index-core sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 109.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.0/362.0 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.0/165.0 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 15.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take 

# Parsing

In [ ]:
import os
import re
from llama_parse import LlamaParse

os.environ["LLAMA_CLOUD_API_KEY"] = "llx-ji7MHVzFE3gzv6cX5NdYgvollzGT6cYY2xi82LD98AXMiuxg"

def parse_pdf_with_metadata(file_path):
    parser = LlamaParse(
        result_type="markdown",
        verbose=True,
        language="en",
    )

    documents = parser.load_data(file_path)

    for i, doc in enumerate(documents):
        doc.metadata["file_name"] = os.path.basename(file_path)
        doc.metadata["page"] = i + 1

        cleaned_text = re.sub(r'<br\s*/?>', ' ', doc.text, flags=re.IGNORECASE)
        doc.set_content(cleaned_text)

    return documents

pdf_file = "General classification.pdf"
parsed_docs = parse_pdf_with_metadata(pdf_file)

target_page = 31
doc = parsed_docs[target_page]

print(f"--- الميتا داتا ---")
print(f"اسم الملف: {doc.metadata['file_name']}")
print(f"رقم الصفحة: {doc.metadata['page']}")
print(f"--- النص/الجدول (Markdown) ---")
print(doc.text)

Started parsing the file under job_id 30774387-5cda-422c-a643-11781de6ab43
--- الميتا داتا ---
اسم الملف: General classification.pdf
رقم الصفحة: 32
--- النص/الجدول (Markdown) ---

# APPENDICES

# APPENDIX 3

## COMPARISON OF 1999 WHO AND 2003 ADA DIAGNOSTIC CRITERIA

# Comparison of 1999 and 2003 Diagnostic Criteria

|                                  | WHO 1999                                                             | ADA 2003                                                                 |
| -------------------------------- | -------------------------------------------------------------------- | ------------------------------------------------------------------------ |
| **Diabetes** Fasting glucose | ≥7.0mmol/l **or** ≥11.1mmol/l                                | ≥7.0mmol/l **or** ≥11.1mmol/l                                    |
| 2–h glucose\*                    | ≥11.1mmol/l                                                          | ≥11.1mmol/l                                 

# Fixed Chunking

In [ ]:
from langchain_core.documents import Document as LangchainDocument
from langchain_text_splitters import CharacterTextSplitter

langchain_docs = [
    LangchainDocument(
        page_content=doc.text,
        metadata=doc.metadata
    )
    for doc in parsed_docs
]

text_splitter = CharacterTextSplitter(
    separator="",
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(langchain_docs)
print(f"إجمالي عدد الـ Chunks الناتجة: {len(chunks)}")
print(chunks[80].page_content)
print(chunks[80].metadata)

إجمالي عدد الـ Chunks الناتجة: 113
HO diagnostic criteria for IFG**

The Group was mindful of the implications of having different WHO and ADA criteria for IFG. The ADA recommendations are targeted to health care providers in one country compared with the global WHO recommendations. It was also noted that other significant discrepancies already exist between the ADA and WHO recommendations including the method for diagnosing diabetes eg fasting plasma glucose versus oral glucose tolerance test. Although these different recommendations for IFG may lead to some confusion initially it should stimulate research to provide data to resolve the discrepancy and other issues associated with defining cut-points for fasting plasma glucose.
{'file_name': 'General classification.pdf', 'page': 24}


# Recursive Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document as LangchainDocument

# Ensure langchain_docs is defined for this cell
langchain_docs = [
    LangchainDocument(
        page_content=doc.text,
        metadata=doc.metadata
    )
    for doc in parsed_docs
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(langchain_docs)

print(f"إجمالي عدد الـ Chunks الناتجة: {len(chunks)}")

if len(chunks) > 80:
    print("\n--- عينة من Chunk رقم 80 ---")
    print(chunks[80].page_content)
    print("\n--- الميتا داتا الخاصة به ---")
    print(chunks[80].metadata)
else:
    print(f"عدد الـ Chunks الكلي ({len(chunks)}) أقل من 80، جرب تطبّق على رقم أصغر مثل chunks[0]")

إجمالي عدد الـ Chunks الناتجة: 135

--- عينة من Chunk رقم 80 ---
- Concordance of IFG and IGT

- Risk profile of individuals identified with IFG

- Economic considerations and cost implications

- Implications for health services and policy

Each of these points is considered below.

## **Outcomes**

As reviewed above, available data do not point to a specific and consistent cut-point for adverse cardiovascular outcomes or mortality.

## **Incident diabetes**

--- الميتا داتا الخاصة به ---
{'file_name': 'General classification.pdf', 'page': 21}


# Embedding Models

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-m3")

texts = [chunk.page_content if hasattr(chunk, 'page_content') else chunk for chunk in chunks]

embeddings = model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

(135, 1024)


In [ ]:
import os
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-large")

texts = [chunk.page_content if hasattr(chunk, 'page_content') else chunk for chunk in chunks]

embeddings = model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(embeddings.shape)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

(135, 1024)


# **Book 2**

In [ ]:
import os
import re
from llama_parse import LlamaParse

os.environ["LLAMA_CLOUD_API_KEY"] = "llx-ji7MHVzFE3gzv6cX5NdYgvollzGT6cYY2xi82LD98AXMiuxg"

def parse_pdf_with_metadata(file_path):
    parser = LlamaParse(
        result_type="markdown",
        verbose=True,
        language="en",
    )

    documents = parser.load_data(file_path)

    for i, doc in enumerate(documents):
        doc.metadata["file_name"] = os.path.basename(file_path)
        doc.metadata["page"] = i + 1

        cleaned_text = re.sub(r'<br\s*/?>', ' ', doc.text, flags=re.IGNORECASE)
        doc.set_content(cleaned_text)

    return documents

pdf_file = "Type 2_20.1.pdf"
parsed_docs = parse_pdf_with_metadata(pdf_file)

target_page = 8
doc = parsed_docs[target_page]

print(f"--- الميتا داتا ---")
print(f"اسم الملف: {doc.metadata['file_name']}")
print(f"رقم الصفحة: {doc.metadata['page']}")
print(f"--- النص/الجدول (Markdown) ---")
print(doc.text)

Error while parsing the file 'Type 2_20.1.pdf': [Errno 2] No such file or directory: '/content/Type 2_20.1.pdf'


IndexError: list index out of range

**Book 3**

In [ ]:
import os
import re
from llama_parse import LlamaParse

os.environ["LLAMA_CLOUD_API_KEY"] = "llx-ji7MHVzFE3gzv6cX5NdYgvollzGT6cYY2xi82LD98AXMiuxg"

def parse_pdf_with_metadata(file_path):
    parser = LlamaParse(
        result_type="markdown",
        verbose=True,
        language="en",
    )

    documents = parser.load_data(file_path)

    for i, doc in enumerate(documents):
        doc.metadata["file_name"] = os.path.basename(file_path)
        doc.metadata["page"] = i + 1

        cleaned_text = re.sub(r'<br\s*/?>', ' ', doc.text, flags=re.IGNORECASE)
        doc.set_content(cleaned_text)

    return documents

pdf_file = "Type 2_3.1.pdf"
parsed_docs = parse_pdf_with_metadata(pdf_file)

target_page = 35
doc = parsed_docs[target_page]

print(f"--- الميتا داتا ---")
print(f"اسم الملف: {doc.metadata['file_name']}")
print(f"رقم الصفحة: {doc.metadata['page']}")
print(f"--- النص/الجدول (Markdown) ---")
print(doc.text)

Started parsing the file under job_id 656c5aaa-8179-456a-a186-6849efb68cd9
--- الميتا داتا ---
اسم الملف: Type 2_3.1.pdf
رقم الصفحة: 36
--- النص/الجدول (Markdown) ---

**Table 1 – Biochemical criteria (venous plasma) for the diagnosis of diabetes, impaired glucose tolerance and impaired fasting glucose or impaired fasting glucose***

|                                                                                                                                          | Glucose concentration, mmol l¹ (mg dl¹) (Venous plasma) |
| ---------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------- |
| **Diabetes mellitus:**<br/>Fasting *and/or*<br/>2-h post glucose load                                                                    | => 7.0 (=> 126)<br/>=> 11.1 (=> 200)                    |
| **Impaired glucose tolerance (IGT)**<br/>Fasting (if measured)<br

# Day 2 Hackathon

**Third Embedding Model**

In [ ]:
!pip install -U bitsandbytes accelerate

using model Qwen/Qwen3-Embedding-4B

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import BitsAndBytesConfig

# إعداد ضغط 4-bit لتوفير الذاكرة
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = SentenceTransformer(
     "Qwen/Qwen3-Embedding-4B",
     device="cuda",
     trust_remote_code=True,
     model_kwargs={
         "quantization_config": quantization_config,
         "device_map": "auto"
     }
)

texts = [chunk.page_content if hasattr(chunk, 'page_content') else chunk for chunk in chunks]

embeddings = model.encode(
    texts,
    batch_size=2,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/30.4k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Batches:   0%|          | 0/53 [00:00<?, ?it/s]

(105, 2560)


using model BAAI/bge-m3

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import BitsAndBytesConfig

# إعداد ضغط 4-bit لتوفير الذاكرة العشوائية (VRAM)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

# تحميل نموذج BAAI/bge-m3 مع تطبيق إعدادات الضغط والتشغيل على الكارت (CUDA)
model = SentenceTransformer(
    "BAAI/bge-m3",
    device="cuda",
    trust_remote_code=True,
    model_kwargs={
        "quantization_config": quantization_config,
        "device_map": "auto"
    }
)

# تجهيز النصوص
texts = [chunk.page_content if hasattr(chunk, 'page_content') else chunk for chunk in chunks]

# استخراج الـ Embeddings (تم ضبط الـ batch_size ليناسب الاستهلاك)
embeddings = model.encode(
    texts,
    batch_size=4,  # يمكنك رفع أو خفض هذا الرقم بناءً على مساحة الـ VRAM المتاحة لديك
    show_progress_bar=True,
    normalize_embeddings=True
)

print(embeddings.shape)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

(105, 1024)


Chroma **DB**

In [ ]:
!pip install -U chromadb

**Delete Old Collection**

In [ ]:
collection_name = "My_pdf_collection"

# 1. مسح الـ Collection القديمة بشكل صريح ورؤية الأخطاء إن وجدت
try:
    client.delete_collection(collection_name)
    print("Deleted old collection successfully.")
except Exception as e:
    print(f"Could not delete collection: {e}")

# 2. إنشاء مجموعة جديدة مع ضمان استخدام get_or_create لتفادي InternalError
collection = client.get_or_create_collection(name=collection_name)

# 3. إذا كانت المجموعة محتفظة بالبيانات القديمة، نقوم بمسح كل البيانات منها لتصفييرها (تصبح 0)
if collection.count() > 0:
    existing_ids = collection.get()["ids"]
    if existing_ids:
        collection.delete(ids=existing_ids)

# 4. إضافة البيانات الجديدة (الـ 105 عناصر)
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings.tolist(), # Convert numpy array to list
)

# 5. التأكد من العدد النهائي
print("Total count in collection:", collection.count())

Could not delete collection: name 'client' is not defined


NameError: name 'client' is not defined

**DB Creation**

In [ ]:
import chromadb

# 1. إنشاء قاعدة بيانات محلية تُحفظ على القرص داخل مجلد باسم "chroma_db"
client = chromadb.PersistentClient(path="./chroma_db")

# 2. إنشاء مجموعة (Collection) جديدة لتخزين بيانات المستند
collection = client.get_or_create_collection(
    name="My_pdf_collection",
    metadata={"hnsw:space": "cosine"}  # قياس التشابه باستخدام Cosine Similarity
)

# 3. تجهيز النصوص والـ IDs والـ Metadata
documents = []
metadatas = []
ids = []

for idx, chunk in enumerate(chunks):
    # دعم النصوص إذا كانت كائنات Document أو نصوص عادية
    if hasattr(chunk, 'page_content'):
        text = chunk.page_content
        meta = chunk.metadata if hasattr(chunk, 'metadata') else {}
    else:
        text = chunk
        meta = {}

    documents.append(text)
    metadatas.append(meta)
    ids.append(f"doc_chunk_{idx}")

# 4. إدخال النصوص والـ Embeddings والـ Metadata دفعة واحدة في ChromaDB
collection.add(
    ids=ids,
    embeddings=embeddings.tolist(),  # تحويل numpy array إلى list
    documents=documents,
    metadatas=metadatas
)

print(f"تم تخزين {collection.count()} قطعة (Chunk) بنجاح داخل ChromaDB!")

InvalidArgumentError: Collection expecting embedding with dimension of 2560, got 1024

the new collection

In [ ]:
import chromadb

# 1. الاتصال بقاعدة البيانات المحلية على القرص
client = chromadb.PersistentClient(path="./chroma_db")

# 2. إنشاء مجموعة جديدة كلياً (Collection) لتجنب تعارض حجم الـ Embeddings (1024)
collection = client.get_or_create_collection(
    name="My_pdf_collection_v2",
    metadata={"hnsw:space": "cosine"}  # قياس التشابه باستخدام Cosine Similarity
)

# 3. تجهيز النصوص والـ IDs والـ Metadata من الـ chunks
documents = []
metadatas = []
ids = []

for idx, chunk in enumerate(chunks):
    # دعم النصوص إذا كانت كائنات Document أو نصوص عادية
    if hasattr(chunk, 'page_content'):
        text = chunk.page_content
        meta = chunk.metadata if hasattr(chunk, 'metadata') else {}
    else:
        text = chunk
        meta = {}

    documents.append(text)
    metadatas.append(meta)
    ids.append(f"doc_chunk_{idx}")

# 4. إدخال النصوص والـ Embeddings والـ Metadata دفعة واحدة في ChromaDB
# (تأكد أن متغير 'embeddings' ناتج من نموذج BAAI/bge-m3 وشغال تمام)
collection.add(
    ids=ids,
    embeddings=embeddings.tolist(),  # تحويل numpy array إلى list
    documents=documents,
    metadatas=metadatas
)

# 5. التأكد من نجاح التخزين وعرض العدد الإجمالي
print(f"تم تخزين {collection.count()} قطعة (Chunk) بنجاح داخل ChromaDB (مجموعة My_pdf_collection_v2)!")

تم تخزين 105 قطعة (Chunk) بنجاح داخل ChromaDB (مجموعة My_pdf_collection_v2)!


Querying ChromaDB

In [ ]:
# 1. Define your search query
query_text = "What key differences exist between the ADA and WHO recommendations regarding glucose tolerance classification?"  # Replace with your actual question

# 2. Encode the query using the SAME model (crucial for accurate embeddings!)
query_embedding = model.encode(
    [query_text],
    normalize_embeddings=True
)

# 3. Query the ChromaDB collection to find the most relevant chunks
results = collection.query(
    query_embeddings=query_embedding.tolist(),  # Convert numpy array to list
    n_results=5                                 # Number of top relevant chunks to retrieve
)

# 4. Display the retrieved results
print("--- Search Results ---")
for i, (doc, meta, distance) in enumerate(zip(results['documents'][0], results['metadatas'][0], results['distances'][0])):
    print(f"\nResult {i+1}:")
    print(f"Distance Score (Cosine): {distance:.4f}")
    print(f"Metadata: {meta}")
    print(f"Content:\n{doc}")

--- Search Results ---

Result 1:
Distance Score (Cosine): 0.2762
Metadata: {'file_name': 'General classification.pdf', 'page': 24}
Content:
## **Implications of not changing the current WHO diagnostic criteria for IFG**

The Group was mindful of the implications of having different WHO and ADA criteria for IFG. The ADA recommendations are targeted to health care providers in one country compared with the global WHO recommendations. It was also noted that other significant discrepancies already exist between the ADA and WHO recommendations including the method for diagnosing diabetes eg fasting plasma glucose versus oral glucose tolerance test. Although these different recommendations for IFG may lead to some confusion initially it should stimulate research to provide data to resolve the discrepancy and other issues associated with defining cut-points for fasting plasma glucose.

Result 2:
Distance Score (Cosine): 0.2890
Metadata: {'file_name': 'General classification.pdf', 'page': 6}


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Get the texts of the retrieved documents
retrieved_document_texts = results['documents'][0]

# Encode the retrieved document texts using the same embed_model
# Adding 'passage: ' prefix as used for indexing
retrieved_embeddings = model.encode(
    [f"passage: {text}" for text in retrieved_document_texts],
    normalize_embeddings=True
)

# Calculate cosine similarity between the query embedding and each retrieved document embedding
similarity_scores = cosine_similarity(query_embedding, retrieved_embeddings)

print("\n" + "=" * 60)
print("Cosine Similarity Scores (Query vs. Retrieved Documents):")
print("=" * 60)
for i, score in enumerate(similarity_scores[0]):
    print(f"Document {i+1}: {score:.4f}")


Cosine Similarity Scores (Query vs. Retrieved Documents):
Document 1: 0.7195
Document 2: 0.7196
Document 3: 0.6835
Document 4: 0.6575
Document 5: 0.6506


**Top-k Value Selection & Justification**

In [ ]:
import pandas as pd


def select_top_k(target_k: int = 5, avg_chunk_tokens: int = 250):
    max_context_window = 4096
    total_tokens = target_k * avg_chunk_tokens

    justification = {
        "chosen_k": target_k,
        "estimated_retrieval_tokens": total_tokens,
        "context_window_usage_pct": round(
            (total_tokens / max_context_window) * 100, 2
        ),
        "rationale": f"تم اختيار k={target_k} لتسليم سياق كامل (~{total_tokens} tokens) دون تجاوز حد الـ Context Window أو تسريع وقت الاستجابة.",
    }

    print(pd.DataFrame([justification]))
    return target_k


TOP_K = select_top_k(target_k=5)

   chosen_k  estimated_retrieval_tokens  context_window_usage_pct  \
0         5                        1250                     30.52   

                                           rationale  
0  تم اختيار k=5 لتسليم سياق كامل (~1250 tokens) ...  


**Chunk Size & Overlap Experimenting**

In [ ]:
import sys
!{sys.executable} -m pip install -U langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
# قائمة التجارب
experiments = [
    {"chunk_size": 500, "overlap": 50},
    {"chunk_size": 800, "overlap": 100},
    {"chunk_size": 1000, "overlap": 200},
]

embedding_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large"
)


def run_chunking_experiment(raw_docs, exp_config):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=exp_config["chunk_size"],
        chunk_overlap=exp_config["overlap"],
    )
    chunks = splitter.split_documents(raw_docs)

    # إنشاء قاعدة بيانات مؤقتة للتجربة
    vector_db = Chroma.from_documents(chunks, embedding_model)
    total_chunks = len(chunks)

    print(
        f"Exp [Size: {exp_config['chunk_size']}, Overlap: {exp_config['overlap']}] -> Generated {total_chunks} chunks"
    )
    return vector_db

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

**Test Set Generation (5-10 Questions)**

In [ ]:
import json

test_dataset = [
    {
        "id": 1,
        "query": "What are the WHO criteria for defining IFG?",
        "expected_chunk_ids": ["doc_chunk_0", "doc_chunk_1"],
    },
    {
        "id": 2,
        "query": "What is the difference between ADA and WHO fasting glucose cut-points?",
        "expected_chunk_ids": ["doc_chunk_5"],
    },
    {
        "id": 3,
        "query": "How is asymptomatic diabetes diagnosed according to WHO?",
        "expected_chunk_ids": ["doc_chunk_12", "doc_chunk_13"],
    },
    {
        "id": 4,
        "query": "What are the guidelines for Impaired Glucose Tolerance?",
        "expected_chunk_ids": ["doc_chunk_20"],
    },
    {
        "id": 5,
        "query": "What is the recommended threshold for IFG set by ADA in 2003?",
        "expected_chunk_ids": ["doc_chunk_3"],
    },
]

# حفظ مجموعة الاختبار
with open("rag_test_set.json", "w", encoding="utf-8") as f:
    json.dump(test_dataset, f, ensure_ascii=False, indent=4)

print("Saved 5 test cases successfully.")

Saved 5 test cases successfully.


Precision@k computed for at least two embedding models

In [ ]:
import numpy as np


def compute_precision_at_k(retrieved_ids, expected_ids, k):
    retrieved_top_k = retrieved_ids[:k]
    relevant_retrieved = set(retrieved_top_k).intersection(set(expected_ids))
    return len(relevant_retrieved) / k


# خوارزمية التقييم للموديل
def evaluate_model(vector_store, test_set, k=5):
    precisions = []
    for sample in test_set:
        results = vector_store.similarity_search(sample["query"], k=k)
        retrieved_ids = [doc.metadata.get("chunk_id", "") for doc in results]
        precision = compute_precision_at_k(
            retrieved_ids, sample["expected_chunk_ids"], k
        )
        precisions.append(precision)
    return np.mean(precisions)


# مثال للمقارنة بين موديلين
# precision_model_1 = evaluate_model(db_model_bge, test_dataset, k=5)
# precision_model_2 = evaluate_model(db_model_minilm, test_dataset, k=5)

**Retrieval Scores Logging & Comparison**

In [ ]:
import pandas as pd

# 1. تعديل الدالة لتتوافق مع ChromaDB مباشرة
def compare_chroma_collections(query, collection_1, collection_2, model_1, model_2, k=3):
    # ترميز السؤال للنموذج الأول
    emb_1 = model_1.encode([query], normalize_embeddings=True).tolist()
    res_1 = collection_1.query(query_embeddings=emb_1, n_results=k)

    # ترميز السؤال للنموذج الثاني
    emb_2 = model_2.encode([query], normalize_embeddings=True).tolist()
    res_2 = collection_2.query(query_embeddings=emb_2, n_results=k)

    logs = []
    for i in range(k):
        logs.append({
            "Rank": i + 1,
            "Model_1_Score": round(res_1['distances'][0][i], 4),
            "Model_1_Text": res_1['documents'][0][i][:60] + "...",
            "Model_2_Score": round(res_2['distances'][0][i], 4),
            "Model_2_Text": res_2['documents'][0][i][:60] + "...",
        })

    return pd.DataFrame(logs)

# 2. سؤال البحث
query = "What are the WHO criteria for defining IFG?"

# 3. استدعاء الدالة (تأكد من تمرير أسماء الـ collections والـ models الصحيحة لديك)
# استبدل collection_1 و collection_2 بأسماء المجموعات التي أنشأتها (مثل collection و My_pdf_collection_v2)
df_result = compare_chroma_collections(
    query=query,
    collection_1=collection,        # المجموعة الأولى
    collection_2=collection,        # المجموعة الثانية (أو أي collection أخرى تريد مقارنتها)
    model_1=model,                  # نموذج المرتكز الأول
    model_2=model,                  # نموذج المرتكز الثاني
    k=3
)

# 4. عرض الجدول
display(df_result)

,Rank,Model_1_Score,Model_1_Text,Model_2_Score,Model_2_Text
0,1,0.3609,Studies have reported that the cardiovascular ...,0.3609,Studies have reported that the cardiovascular ...
1,2,0.3715,**Figure 3. Comparison of prevalence of IFG di...,0.3715,**Figure 3. Comparison of prevalence of IFG di...
2,3,0.3760,One of the reasons cited by the ADA Committee ...,0.3760,One of the reasons cited by the ADA Committee ...


In [ ]:
pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 80.9 MB/s eta 0:00:00


Retrieved chunks visible before generation (even in a basic UI)

In [ ]:
import streamlit as st

st.title("Medical RAG System with Retrieval Preview")

query_text = st.text_input("أدخل سؤالك الطبي:", "What are the blood glucose thresholds for diagnosing diabetes?")

if st.button("بحث واسترجاع (Retrieve)"):
    # 1. الاسترجاع من ChromaDB
    query_embedding = model.encode([query_text], normalize_embeddings=True)
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=4
    )

    # 2. عرض القطع المسترجعة بوضوح للمستخدم قبل التوليد
    st.subheader("📌 القطع المسترجعة من المستند (Retrieved Chunks Preview):")

    retrieved_docs = results['documents'][0]
    retrieved_metas = results['metadatas'][0]
    retrieved_distances = results['distances'][0]

    for i, (doc, meta, dist) in enumerate(zip(retrieved_docs, retrieved_metas, retrieved_distances)):
        with st.expander(f"Result {i+1} | Page: {meta.get('page', 'N/A')} | Distance: {dist:.4f}"):
            st.write(doc)

    # 3. زر لتأكيد الإرسال للـ LLM أو التوليد التلقائي
    st.success("تم استرجاع السياق بنجاح، وجاهز للإرسال للنموذج لتوليد الإجابة!")

2026-08-18 00:54:33.920 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-18 00:54:34.421 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-08-18 00:54:34.424 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-18 00:54:34.430 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-18 00:54:34.433 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-18 00:54:34.435 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-18 00:54:34.440 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-18 00:54:34.445 Session state does not 

In [ ]:
%%writefile app.py
import streamlit as st

st.title("My Medical RAG App")
st.write("مرحباً بك في تطبيق الـ RAG الطبي الخاص بك!")

query = st.text_input("أدخل سؤالك:")
if st.button("بحث"):
    st.success(f"جاري البحث عن: {query}")

Writing app.py


url

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501



⠙⠹⠸⠼⠴⠦⠧⠇⠏Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 2026-08-18 00:56:01.724 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.82.74.31:8501

y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏your url is: https://good-bananas-jog.loca.lt
y
  Stopping...
^C


In [ ]:
# 1. التأكد من تثبيت localtunnel
!npm install -g localtunnel > /dev/null 2>&1

# 2. تشغيل تطبيق Streamlit في الخلفية
!streamlit run app.py &

# 3. انتظار ثانية حتى يعمل السيرفر، ثم فتح النفق عبر localtunnel مع طباعة الرابط بوضوح
import time
import subprocess

# تشغيل localtunnel في الخلفية وقراءة الناتج لاستخراج الرابط
process = subprocess.Popen(["npx", "localtunnel", "--port", "8501"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# طباعة الرابط الذي سيظهر
print("جاري إنشاء الرابط...")
for line in process.stdout:
    if "url" in line.lower() or "https://" in line:
        print(line.strip())
        break



2026-08-18 01:04:34.260 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.82.74.31:8501

  Stopping...
جاري إنشاء الرابط...
your url is: https://whole-foxes-bow.loca.lt


In [ ]:
!npx localtunnel --port 8501


⠙⠹⠸⠼⠴⠦your url is: https://shaggy-laws-punch.loca.lt
^C


**Streamlit UI for Visualizing Retrieved Chunks**

In [ ]:
!pip install streamlit
import streamlit as st

st.title("RAG Retrieval Visualizer")

query = st.text_input("Enter your question:")

if st.button("Retrieve Context"):
    if query:
        # استدعاء الـ ChromaDB المجهزة
        results = collection.query(query_texts=[query], n_results=3)

        st.subheader("Retrieved Chunks (Before Generation)")

        for idx, (doc, score, meta) in enumerate(
            zip(
                results["documents"][0],
                results["distances"][0],
                results["metadatas"][0],
            )
        ):
            with st.expander(f"Chunk {idx+1} | Score: {round(score, 4)}"):
                st.write(f"**Source Page:** {meta.get('page', 'N/A')}")
                st.write(f"**Content:** {doc}")

2026-08-17 18:12:42.708 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 18:12:42.709 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 18:12:42.711 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 18:12:42.712 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 18:12:42.713 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 18:12:42.714 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 18:12:42.716 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 18:12:42.717 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar